# nb4b — Phase 2 (Bước 1): Điền nhãn adjudication bằng LLM (TypeSafe Choice)

Notebook **CPU-only** (không GPU, không train): gọi API TypeSafe (primitive **Choice**, model
`jev-latest`) để điền nhãn A/B/C/D cho 100 mẫu trong `adjudication_samples.json` (output của
nb4), xuất `adjudication_filled.json` **giữ nguyên schema** → upload làm Input → chạy lại nb4
từ đầu (cell §6 của nb4 tự nhận diện và tính noise rate + Wilson CI + sensitivity).

Workflow: `nb4 (xuất samples) → nb4b (điền nhãn) → upload output → nb4 (re-run)`.

**Phương pháp — LLM-assisted adjudication**: nhãn do model đề xuất (API TypeSafe không có tham
số temperature; mỗi nhãn lưu kèm `llm_confidence` + `llm_probabilities` đầy đủ để soát).
**Bắt buộc** người soát trước khi dùng:

- toàn bộ nhãn **B/D** — nhóm quyết định ngưỡng nhiễu > 8% (pre-registered);
- mọi mẫu có `llm_confidence` < `REVIEW_CONF_THRESHOLD`;
- 5 mẫu ngẫu nhiên (seed 42) để ước lượng tỷ lệ đồng thuận.

Nhãn adjudication chỉ dùng để ước lượng nhiễu pseudo-gold + sensitivity, **không** fit tham số
model/pipeline nào (`DESIGN.md` §9); ghi tỷ lệ đồng thuận sau soát vào REPORT.

**Vận hành trên Kaggle (~10 phút, CPU)**:

1. Attach Input: dataset chứa `adjudication_samples.json` (output của lần chạy nb4).
2. Add-ons → Secrets → thêm secret tên `TYPESAFE_API_KEY` (hoặc env, hoặc nhập tay khi được hỏi).
3. Lần đầu: giữ `PILOT_N = 10` → Run All → soi nhãn pilot in ở output (gate: ≥ 8/10 khớp phán
   đoán của bạn; case tham chiếu `adj_00423` kỳ vọng **A** — 3/4 edit là typo telex thật
   `wuy→quy, kinj→kinh, Nsm→Nam`, riêng edit `H→KHĐT` là phi chính tả nhưng chỉ lẻ tẻ).
4. Đạt → đặt `PILOT_N = 0` → Run All lại từ đầu → soát `review_queue` in cuối notebook → sửa
   nhãn sai trực tiếp trong `adjudication_filled.json` (điền `label_note` nếu cần).
5. Save Version → tải `adjudication_filled.json` → upload lên Kaggle Dataset → attach vào nb4 →
   Run All (CONSISTENCY CHECK của nb4 vẫn phải PASS; §6 in noise + sensitivity).


In [ ]:
import os
import sys
import json
import time
import random
import getpass
import datetime
import subprocess
import collections
from pathlib import Path

NOTEBOOK = 'nb4b_adjudication_llm_fill'
VERSION = 'v1'
RUN_STAMP = datetime.datetime.now().isoformat(timespec='seconds')

# Buckets cấu hình — mọi hằng số gom một chỗ (DESIGN.md §7)
MODEL = 'jev-latest'
PILOT_N = 10                  # >0: chỉ chạy N mẫu đầu → adjudication_pilot.json (soi tay trước khi chạy đủ 100)
REVIEW_CONF_THRESHOLD = 0.85  # confidence dưới ngưỡng → bắt buộc soát tay
CONCURRENCY = 1               # thread-safety của SDK chưa kiểm chứng → giữ 1 (nâng 2-4 nếu cần nhanh)
MAX_RETRIES = 4               # số lần thử lại cho 429/529/5xx/lỗi mạng
RETRY_DELAYS = [2, 4, 8, 16]  # backoff (giây)
SEED = 42
INPUT_NAME = 'adjudication_samples.json'
OUTPUT_NAME = 'adjudication_filled.json'   # PILOT_N > 0 → adjudication_pilot.json
REVIEW_NAME = 'review_queue.json'
ENDPOINT = 'https://api.typesafe.ai/v1/systemone'
VALID_LABELS = ('A', 'B', 'C', 'D')

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('./out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_input(name, required=True):
    """Tự dò Input: quét đệ quy /kaggle/input (Kaggle), fallback ./data (chạy cục bộ)."""
    for root in (Path('/kaggle/input'), Path('./data')):
        if root.is_dir():
            hits = sorted(root.rglob(name))
            if hits:
                return hits[0]
    if required:
        raise FileNotFoundError(
            f'Không tìm thấy {name} — attach Input chứa adjudication_samples.json (output của nb4).')
    return None


INPUT_FILE = find_input(INPUT_NAME)
RESUME_FILE = find_input(OUTPUT_NAME, required=False)   # output lần trước nếu được attach → resume
print(f'Input : {INPUT_FILE}')
if RESUME_FILE is not None:
    print(f'Resume: {RESUME_FILE} (bỏ qua các mẫu đã có nhãn)')


def resolve_api_key():
    """Ưu tiên Kaggle Secrets → env TYPESAFE_API_KEY → nhập tay. Không in key ra log."""
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('TYPESAFE_API_KEY')
        if key:
            return key, 'kaggle-secrets'
    except Exception:
        pass
    key = os.environ.get('TYPESAFE_API_KEY')
    if key:
        return key, 'env'
    return getpass.getpass('Nhập TYPESAFE_API_KEY (input ẩn): '), 'manual'


API_KEY, KEY_SOURCE = resolve_api_key()
assert API_KEY and API_KEY.strip(), 'Thiếu TYPESAFE_API_KEY — thêm Kaggle Secret/env hoặc nhập tay.'
os.environ['TYPESAFE_API_KEY'] = API_KEY   # SDK đọc key từ env
print(f'API key: có (nguồn: {KEY_SOURCE})')

# SDK typesafe-sdk (Python >= 3.10); lỗi install/import → fallback HTTP urllib (cell API)
try:
    import typesafe_sdk  # noqa: F401
    USE_SDK = True
except ImportError:
    print('Đang cài typesafe-sdk ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'typesafe-sdk'], check=False)
    try:
        import typesafe_sdk  # noqa: F401
        USE_SDK = True
    except ImportError:
        USE_SDK = False
backend = 'typesafe-sdk' if USE_SDK else 'urllib (HTTP fallback)'
print(f'Backend: {backend} · model={MODEL}')
print(f'Output dir: {OUTPUT_DIR} · {NOTEBOOK} {VERSION} · {RUN_STAMP}')


In [ ]:
# Câu hỏi Choice duy nhất: nhãn A/B/C/D theo rubric nb4 §6.
# Lưu ý: KHÔNG nhúng ví dụ trùng dữ liệu thật vào criteria (tránh bias nhãn, vd adj_00423).
TS_INSTRUCTIONS = {
    'ngu_canh': ('Đánh giá chất lượng nhãn của bộ dữ liệu sửa lỗi chính tả tiếng Việt: '
                 '`text` là câu gốc, `corrected_text` là bản sửa chuẩn của dataset, '
                 '`src_marked`/`tgt_marked` bọc [các vị trí edit], `edit_blocks` liệt kê từng edit.'),
    'cau_hoi': 'Cặp câu này thuộc nhãn nào?',
    'uu_tien': ('Đánh giá TOÀN BỘ cặp câu, không xét từng edit rời. '
                'Đa số edit là lỗi chính tả thật, lẻ tẻ 1-2 edit tên riêng/định dạng → A. '
                'Đa số edit là tên riêng/số/viết tắt/định dạng → C. '
                'Hai vế không cùng một câu → D. '
                "Vế source vốn đúng mà bản 'sửa' sai hoặc sửa oan → B."),
}
TS_CRITERIA = {
    'A': 'Hai vế song song; các edit là lỗi chính tả thật (typo, sai/thiếu/thừa dấu, '
         'gõ telex/vni sai: wuy→quy, kinj→kinh) → nhãn OK',
    'B': 'Vế source vốn đã đúng / bản sửa sai (sửa oan, đổi sang từ khác nghĩa, viết tắt '
         'bị thay bằng từ đầy đủ như QH→Quốc hội) → nhãn NHIỄU',
    'C': 'Edit không phải lỗi chính tả: số liệu, viết tắt, tên riêng, định dạng → không tính là nhiễu',
    'D': 'Hai vế không song song / align hỏng → tính là nhiễu',
}


def build_state(s):
    """State object cho TypeSafe: chỉ giữ trường cần thiết để phán xét."""
    return {
        'text': s['text'],
        'corrected_text': s['corrected_text'],
        'src_marked': s['src_marked'],
        'tgt_marked': s['tgt_marked'],
        'edit_blocks': [{'src': b['src'], 'tgt': b['tgt'], 'type': b['type'],
                         'punct_only': b['punct_only']} for b in s['edit_blocks']],
    }


print('Prompt: 1 câu Choice "label" · criteria A/B/C/D · state object 5 trường')


In [ ]:
import urllib.request
import urllib.error


def _extract(resp_dict):
    """Chuẩn hóa answer Choice từ JSON thô (dùng bởi HTTP fallback)."""
    ans = resp_dict['answers']['label']
    lab = str(ans.get('choice', '')).strip().upper()
    if lab not in VALID_LABELS:
        return None, None, f'choice lạ: {ans.get("choice")!r}'
    meta = (float(ans.get('confidence') or 0.0),
            dict(ans.get('probabilities') or {}),
            str(resp_dict.get('model') or MODEL))
    return lab, meta, None


def call_http(sample):
    """Fallback không cần SDK: POST thẳng /v1/systemone (schema theo docs.typesafe.ai/api)."""
    body = json.dumps({
        'state': build_state(sample),
        'model': MODEL,
        'questions': {'label': {'type': 'choice',
                                'instructions': TS_INSTRUCTIONS,
                                'criteria': TS_CRITERIA}},
    }).encode('utf-8')
    req = urllib.request.Request(ENDPOINT, data=body, headers={
        'Content-Type': 'application/json', 'Authorization': f'Bearer {API_KEY}'})
    with urllib.request.urlopen(req, timeout=60) as resp:
        return _extract(json.loads(resp.read().decode('utf-8')))


def _get(obj, name, default=None):
    """Đọc phòng thủ: SDK có thể trả object hoặc dict."""
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


def call_sdk(sample):
    from typesafe_sdk import Choice, TypeSafeClient
    questions = {'label': Choice(instructions=TS_INSTRUCTIONS, criteria=TS_CRITERIA)}
    state = build_state(sample)
    try:
        with TypeSafeClient() as client:   # docs dùng cả 2 kiểu; thử context manager trước
            response = client.system_one(state=state, questions=questions)
    except (TypeError, AttributeError):
        response = TypeSafeClient().system_one(state=state, questions=questions)
    ans = _get(response, 'answers')['label']
    lab = str(_get(ans, 'choice', '')).strip().upper()
    if lab not in VALID_LABELS:
        return None, None, f'choice lạ: {lab!r}'
    meta = (float(_get(ans, 'confidence', 0.0) or 0.0),
            dict(_get(ans, 'probabilities', {}) or {}),
            str(_get(response, 'model', MODEL) or MODEL))
    return lab, meta, None


def call_with_retry(sample):
    """Trả (label, meta=(conf, probs, api_model), error). Backoff cho 429/529/5xx/mạng;
    4xx khác (401/422) dừng sớm vì retry vô ích."""
    fn = call_sdk if USE_SDK else call_http
    last = None
    for attempt in range(MAX_RETRIES):
        try:
            return fn(sample)
        except urllib.error.HTTPError as e:
            last = f'HTTP {e.code}: {e.reason}'
            if e.code not in (429, 529) and e.code < 500:
                break
        except Exception as e:   # URLError, timeout, schema SDK lạ...
            last = f'{type(e).__name__}: {str(e)[:120]}'
        if attempt < MAX_RETRIES - 1:
            time.sleep(RETRY_DELAYS[min(attempt, len(RETRY_DELAYS) - 1)])
    return None, None, last


print('Đã định nghĩa call_sdk/call_http/call_with_retry '
      f'(backend={"sdk" if USE_SDK else "http"}, retries={MAX_RETRIES}).')


In [ ]:
payload = json.loads(Path(INPUT_FILE).read_text(encoding='utf-8'))
samples = payload['samples']
n_total = len(samples)
assert n_total == payload.get('n_samples', n_total), 'payload.samples != n_samples — file lệch schema'

results = {}   # sample_id → sample dict + nhãn (kèm llm_*)
if RESUME_FILE is not None:
    prev = json.loads(Path(RESUME_FILE).read_text(encoding='utf-8'))
    for s in prev.get('samples', []):
        if s.get('label') in VALID_LABELS:
            results[s['sample_id']] = s
    print(f'Resume: giữ {len(results)} nhãn có sẵn.')

todo = [s for s in samples if s['sample_id'] not in results]
if PILOT_N > 0:
    todo = todo[:PILOT_N]
print(f'Gọi TypeSafe cho {len(todo)} mẫu · PILOT_N={PILOT_N} · concurrency={CONCURRENCY}')


def with_label(s, lab, meta):
    return {**s, 'label': lab, 'label_note': '',
            'llm_confidence': meta[0], 'llm_probabilities': meta[1], 'llm_model': meta[2]}


def write_payload(results_map, path, meta=None):
    """Ghi payload giữ nguyên schema đầu vào + nhãn; meta_llm_fill chỉ ở bản cuối."""
    out = []
    for s in samples:
        r = results_map.get(s['sample_id'])
        out.append(r if r is not None else {**s, 'label': None, 'label_note': ''})
    doc = {**payload, 'samples': out}
    if meta:
        doc['meta_llm_fill'] = meta
    Path(path).write_text(json.dumps(doc, ensure_ascii=False, indent=2), encoding='utf-8')


failures = []
t0 = time.time()
if CONCURRENCY <= 1:
    for i, s in enumerate(todo, 1):
        sid = s['sample_id']
        lab, meta, err = call_with_retry(s)
        if lab:
            results[sid] = with_label(s, lab, meta)
        else:
            failures.append((sid, err))
        status = lab if lab else f'FAIL — {err}'
        print(f'  [{i:3d}/{len(todo)}] {sid}: {status}')
        if PILOT_N == 0 and i % 10 == 0:
            write_payload(results, OUTPUT_DIR / OUTPUT_NAME)   # checkpoint định kỳ
else:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    with ThreadPoolExecutor(max_workers=CONCURRENCY) as ex:
        futs = {ex.submit(call_with_retry, s): s for s in todo}
        for i, fut in enumerate(as_completed(futs), 1):
            s = futs[fut]
            sid = s['sample_id']
            lab, meta, err = fut.result()
            if lab:
                results[sid] = with_label(s, lab, meta)
            else:
                failures.append((sid, err))
            status = lab if lab else f'FAIL — {err}'
            print(f'  [{i:3d}/{len(todo)}] {sid}: {status}')

print(f'Xong {len(todo)} mẫu trong {time.time() - t0:.0f}s · ok={len(todo) - len(failures)} · fail={len(failures)}')
if failures:
    preview = '; '.join(f'{sid} ({err})' for sid, err in failures[:10])
    print('FAIL:', preview + ('…' if len(failures) > 10 else ''))


In [ ]:
n_filled = sum(1 for s in samples if s['sample_id'] in results)
api_models = sorted({r['llm_model'] for r in results.values() if r.get('llm_model')})
meta_llm_fill = {
    'filled_by': ('llm-assisted (TypeSafe Choice) — BẮT BUỘC soát tay nhãn B/D + confidence thấp '
                  'trước khi dùng; nhãn chỉ để ước lượng nhiễu pseudo-gold + sensitivity, '
                  'không fit tham số model/pipeline (DESIGN.md §9)'),
    'notebook': NOTEBOOK, 'version': VERSION,
    'model': MODEL,
    'backend': 'typesafe-sdk' if USE_SDK else 'urllib (HTTP fallback)',
    'endpoint_host': 'api.typesafe.ai',
    'api_models': api_models,
    'review_conf_threshold': REVIEW_CONF_THRESHOLD,
    'pilot': PILOT_N > 0,
    'n_total': n_total, 'n_filled': n_filled, 'n_fail': len(failures),
    'seed': SEED,
    'filled_at': RUN_STAMP,
}
out_name = 'adjudication_pilot.json' if PILOT_N > 0 else OUTPUT_NAME
write_payload(results, OUTPUT_DIR / out_name, meta=meta_llm_fill)
print(f'Đã ghi {out_name}: {n_filled}/{n_total} nhãn hợp lệ · fail={len(failures)} · kèm meta_llm_fill')

if PILOT_N > 0:
    print('CHẾ ĐỘ PILOT — file pilot KHÔNG dùng cho nb4. Soi nhãn in phía trên (kỳ vọng tham chiếu: '
          'adj_00044/adj_00140 = A — toàn typo telex; adj_00423 = A vì 3/4 edit là typo thật, '
          'riêng H→KHĐT là phi chính tả lẻ tẻ). Đạt gate ≥8/10 → đặt PILOT_N=0 rồi Run All lại từ đầu.')
else:
    missing = [s['sample_id'] for s in samples if s['sample_id'] not in results]
    if missing:
        preview = ', '.join(missing[:10]) + ('…' if len(missing) > 10 else '')
        raise RuntimeError(
            f'{len(missing)} mẫu chưa có nhãn: {preview}. KHÔNG upload file này cho nb4 (nb4 sẽ từ chối). '
            'Xử lý: xem FAIL phía trên (thử backend HTTP bằng cách ép USE_SDK=False nếu SDK lỗi schema), '
            'rồi Save Version → tải adjudication_filled.json (một phần) → tạo/cập nhật Kaggle Dataset → '
            'attach làm Input → Run All lại (resume giữ các nhãn đã có).')
    print('Đủ nhãn cho toàn bộ mẫu → sang cell review queue.')


In [ ]:
filled = [results[s['sample_id']] for s in samples if s['sample_id'] in results]
cnt = collections.Counter(s['label'] for s in filled)
n_lbl = len(filled)
print(f'=== Tóm tắt nhãn (model={MODEL}, backend={"sdk" if USE_SDK else "http"}) ===')
print('Phân bổ:', dict(cnt))
if n_lbl:
    noise_raw = (cnt['B'] + cnt['D']) / n_lbl
    print(f'Ước lượng nhiễu thô (B+D)/n: {noise_raw:.1%} (n={n_lbl}) — CHƯA qua soát tay; '
          'nb4 sẽ tính Wilson CI + sensitivity chính thức')

# Hàng đợi soát tay: toàn bộ B/D ∪ confidence < ngưỡng ∪ 5 mẫu ngẫu nhiên (seed 42)
rng = random.Random(SEED)
must_review = [s for s in filled
               if s['label'] in ('B', 'D') or (s.get('llm_confidence') or 0.0) < REVIEW_CONF_THRESHOLD]
rest_ids = {s['sample_id'] for s in must_review}
sampled = rng.sample([s for s in filled if s['sample_id'] not in rest_ids],
                     min(5, max(0, n_lbl - len(rest_ids))))
queue, seen = [], set()
for s in must_review + sampled:
    if s['sample_id'] not in seen:
        seen.add(s['sample_id'])
        queue.append(s)
queue.sort(key=lambda s: s.get('test_index', 0))

print()
print(f'=== REVIEW QUEUE: {len(queue)} mẫu cần soát tay (B/D ∪ conf<{REVIEW_CONF_THRESHOLD} ∪ 5 ngẫu nhiên) ===')
for s in queue:
    sid, lab = s['sample_id'], s['label']
    conf = s.get('llm_confidence') or 0.0
    print(f'—— {sid} · nhãn={lab} · conf={conf:.2f}')
    print(f'   SRC: {s["src_marked"]}')
    print(f'   TGT: {s["tgt_marked"]}')

review_payload = {
    'created': RUN_STAMP, 'notebook': NOTEBOOK,
    'threshold': REVIEW_CONF_THRESHOLD, 'n_review': len(queue),
    'rubric': TS_CRITERIA,
    'instructions': [
        'Soát từng mẫu: nhãn sai → sửa trường label trong adjudication_filled.json (+ label_note ghi lý do).',
        'Đếm tỷ lệ đồng thuận trên các mẫu đã soát → ghi vào REPORT (adjudication LLM-assisted).',
        'Sau khi chốt nhãn: Save Version → tải adjudication_filled.json → upload làm Kaggle Dataset '
        '→ attach vào nb4 → Run All từ đầu (§6 tự tính noise + Wilson CI + sensitivity).',
    ],
    'samples': [{'sample_id': s['sample_id'], 'label': s['label'],
                 'confidence': s.get('llm_confidence'),
                 'src_marked': s['src_marked'], 'tgt_marked': s['tgt_marked']} for s in queue],
}
(OUTPUT_DIR / REVIEW_NAME).write_text(json.dumps(review_payload, ensure_ascii=False, indent=2),
                                      encoding='utf-8')
out_file = 'adjudication_pilot.json' if PILOT_N > 0 else OUTPUT_NAME
print()
print(f'Đã ghi {REVIEW_NAME} ({len(queue)} mẫu) vào {OUTPUT_DIR}')
print(f'Các file output: {out_file} · {REVIEW_NAME}')
